# Chapter 8 Assignment — Semantic Search and RAG

**Book:** Hands-On Large Language Models (O'Reilly)  
**Chapter:** 8 — Semantic Search and Retrieval-Augmented Generation  
**Style:** You write all the code. Every exercise has a stub — fill it in.

---

## Table of Contents

| Part | Topic | Exercises |
|------|-------|-----------|
| Part 1 | Dense Retrieval | Ex 1.1 – 1.6 |
| Part 2 | Keyword Search: BM25 | Ex 2.1 – 2.2 |
| Part 3 | Reranking with a Cross-Encoder | Ex 3.1 – 3.4 |
| Part 4 | Retrieval Evaluation Metrics | Ex 4.1 – 4.4 |
| Part 5 | Basic RAG with Ollama | Ex 5.1 – 5.3 |
| Part 6 | LangChain RAG Pipeline | Ex 6.1 – 6.4 |
| Part 7 | Advanced RAG | Ex 7.1 – 7.2 |
| Part 8 | Mini Projects | A, B, C |

---

## What You Will Build

By the end of this notebook you will have a working, end-to-end RAG system that runs **100% locally** — no paid API, no internet required after the first model download. The system will:

1. Embed a text corpus and retrieve relevant chunks by meaning (dense retrieval)
2. Compare that with keyword search (BM25)
3. Rerank candidates with a cross-encoder for higher precision
4. Build an explicit **hybrid BM25 → reranker** pipeline
5. Evaluate retrieval quality with Precision@k, AP, and MAP
6. Feed retrieved context to a local LLM (Ollama) to answer questions with grounding
7. Build the same pipeline with **LangChain** to see how a production framework abstracts it
8. Rewrite queries and decompose complex questions (advanced RAG)

---

## Setup

Install dependencies once, then restart the kernel:

```bash
pip install sentence-transformers faiss-cpu rank_bm25 requests
pip install langchain langchain-community langchain-huggingface  # for Part 6
```

You also need **Ollama** running locally. Install from https://ollama.com, then pull a model:

```bash
ollama pull phi3:mini          # fast, runs on CPU (3.8B)
# OR — if you have a GPU server via SSH tunnel:
# ollama pull llama3.3:70b
```

To use your remote GPU server, open an SSH tunnel before running this notebook:
```bash
ssh -L 11434:localhost:11434 your-server
```
Then set `OLLAMA_BASE_URL` to `"http://localhost:11434"` and `OLLAMA_MODEL` to `"llama3.3:70b"` in the config cell below.

In [207]:
# CONFIG — change these two variables only

# OLLAMA_BASE_URL = "http://localhost:11434"   # local Ollama
OLLAMA_BASE_URL = "http://localhost:11435"   # cloud Ollama
# OLLAMA_MODEL    = "phi3:mini"                # swap to "llama3.3:70b" for SSH tunnel
OLLAMA_MODEL = "phi3.5:latest"

In [208]:
# ============================================================
# IMPORTS AND CORPUS
# ============================================================

import re
import numpy as np
import requests
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# ---- Corpus: Interstellar Wikipedia summary ----
# This is the text archive we will search throughout the notebook.
INTERSTELLAR_TEXT = """
Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.
The screenplay was written by Jonathan Nolan and Christopher Nolan, based on a story developed by Jonathan Nolan.
The film stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, and Michael Caine.
It was produced by Paramount Pictures and Warner Bros. Pictures.
The story follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humanity.
Theoretical physicist Kip Thorne, who won the 2017 Nobel Prize in Physics, served as executive producer and scientific consultant.
Thorne ensured that the depictions of relativity, wormholes, and black holes were as accurate as possible given the narrative constraints.
The film portrays the gravitational time dilation effect predicted by Einstein's general theory of relativity.
On the water planet Miller, one hour equals seven years on Earth due to proximity to the massive black hole Gargantua.
Gargantua is a fictional supermassive rotating black hole with a mass 100 million times that of the Sun.
The visual effects team worked with Thorne to produce scientifically accurate imagery of the black hole.
The result was the first physically accurate simulation of a black hole ever created for a film.
Composer Hans Zimmer created the film's score, using a church organ as the central instrument.
The score is meant to evoke themes of space, time, and the love between parents and children.
Interstellar received positive reviews from critics, who praised its ambition, visual effects, and performances.
The film grossed over 701 million dollars worldwide against a production budget of 165 million dollars.
It won the Academy Award for Best Visual Effects at the 87th Academy Awards.
Cooper, played by McConaughey, is a former NASA pilot and engineer turned farmer.
He discovers a secret NASA facility led by Professor Brand, played by Michael Caine.
Brand reveals that Earth is dying due to a blight that destroys crops and depletes oxygen.
NASA has identified a wormhole near Saturn that leads to a distant galaxy with potentially habitable planets.
The mission is to find a new planet suitable for human colonisation.
Dr. Amelia Brand, played by Anne Hathaway, is a key member of the crew.
The crew visits three candidate planets: Miller, Mann, and Edmunds.
Planet Mann is named after Dr. Mann, played by Matt Damon, who sent a deceptive distress signal.
Dr. Mann falsified data to ensure a rescue mission would come and save him.
Cooper sacrifices himself by falling into Gargantua to give Brand's ship enough thrust to reach Edmunds.
Inside Gargantua, Cooper enters a tesseract constructed by future humans who exist in five dimensions.
The tesseract allows Cooper to communicate across time by manipulating gravitational waves.
He sends the quantum data needed to solve the gravity equation back to his daughter Murph.
Murph uses the equation to enable humanity to leave Earth and populate space stations.
Cooper is eventually ejected from the tesseract and recovered near Saturn.
He wakes up on a space station where an elderly Murph is waiting for him.
The film ends with Cooper setting out to find Brand on Edmunds planet.
Interstellar premiered at the TCL Chinese Theatre in Hollywood on October 26, 2014.
It was released in 70mm film and IMAX formats as well as standard formats.
The IMAX sequences were filmed using large-format cameras for maximum visual impact.
"""

print("Corpus loaded.")
print(f"Total characters: {len(INTERSTELLAR_TEXT)}")

Corpus loaded.
Total characters: 3450


---

## Part 1 — Dense Retrieval

Dense retrieval turns search into geometry. You embed your documents and your query into the same vector space, then find the documents whose vectors are closest to the query vector. This works even when the query uses different words than the document — because embeddings capture *meaning*, not just letters.

The pipeline has four steps:
1. Split the corpus into sentences (your search units)
2. Embed every sentence with a pre-trained model
3. Store the embeddings in a FAISS index for fast search
4. At query time: embed the query, search the index, return top-k sentences

**Reference:** notes §2a–2c

### Exercise 1.1 — Split corpus into sentences

Split `INTERSTELLAR_TEXT` into a list of clean sentences. Strip whitespace and remove empty strings.

**Hint:** Use `re.split` or `.split('\n')` — the corpus above has one sentence per line, so splitting on newlines works well. Strip each sentence and filter out blanks.

**Expected output:**
```
Number of sentences: ~42
First sentence: 'Interstellar is a 2014 epic science fiction film...'
```

In [209]:
def split_into_sentences(text: str) -> list[str]:
    """Split corpus text into a list of non-empty sentences."""
    # YOUR CODE HERE
    sentences = re.split('\n',text)
    return sentences


sentences = split_into_sentences(INTERSTELLAR_TEXT)
print(f"Number of sentences: {len(sentences)}")
print(f"First sentence: {sentences[0]!r}")
print(f"Second sentence: {sentences[1]!r}")
print(f"Third sentence: {sentences[2]!r}")
print(f"Last sentence:  {sentences[-1]!r}")

Number of sentences: 39
First sentence: ''
Second sentence: 'Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.'
Third sentence: 'The screenplay was written by Jonathan Nolan and Christopher Nolan, based on a story developed by Jonathan Nolan.'
Last sentence:  ''


### Exercise 1.2 — Embed with a Sentence Transformer

Load the model `'BAAI/bge-small-en-v1.5'` using `SentenceTransformer` and embed all sentences.

**Why this model?** It is near the top of the MTEB retrieval leaderboard and is small enough to run on CPU in seconds.

**Hint:** `model.encode(sentences)` returns a numpy array of shape `(num_sentences, embedding_dim)`. Convert to `float32` — FAISS requires it.

**Expected output:**
```
Embedding shape: (N, 384)   # N = number of sentences in your corpus
dtype: float32
```

In [210]:
try:
    model = SentenceTransformer('BAAI/bge-small-en-v1.5')
    print("Successful Model loaded.")
    
    # Quick test encode
    emb = model.encode("Hello local machine")
    print(f"Embedding shape: {emb.shape}")
except Exception as e:
    print(f"Error still occurring: {e}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8584.28it/s]


Successful Model loaded.
Embedding shape: (384,)


In [211]:
# Load the embedding model
# YOUR CODE HERE
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# Embed all sentences — shape should be (num_sentences, embedding_dim)
# YOUR CODE HERE
sentence_embeddings = embed_model.encode(sentences)

print(f"Embedding shape: {sentence_embeddings.shape}")
print(f"dtype: {sentence_embeddings.dtype}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19882.01it/s]


Embedding shape: (39, 384)
dtype: float32


### Exercise 1.3 — Build a FAISS index

Create a `faiss.IndexFlatL2` index, then add your sentence embeddings to it.

**Why IndexFlatL2?** It computes exact L2 (Euclidean) distance between the query vector and every stored vector. Perfect for small corpora — no approximation needed.

**Hint:**
- `faiss.IndexFlatL2(embedding_dim)` creates the index
- `index.add(embeddings)` adds vectors — takes a `float32` numpy array
- After adding, `index.ntotal` tells you how many vectors are stored

**Expected output:**
```
FAISS index built. Vectors stored: 42
```

In [212]:
# YOUR CODE HERE
dim = sentence_embeddings.shape[1]
print(f"dim = {dim}")
index = faiss.IndexFlatL2(dim)
index.add(sentence_embeddings)

print(f"FAISS index built. Vectors stored: {index.ntotal}")

dim = 384
FAISS index built. Vectors stored: 39


### Exercise 1.4 — Write a `dense_search` function

Write a function that takes a query string and returns the top-k matching sentences.

**Steps inside the function:**
1. Embed the query with `embed_model.encode([query])` — note the list wrapper
2. Cast to `float32`
3. Call `index.search(query_vec, k)` — returns `(distances, indices)`
4. Use the returned indices to look up the original sentences
5. Return a list of `(distance, sentence)` tuples

**Hint:** `index.search` returns arrays of shape `(1, k)` — use `[0]` to get the first (and only) row.

In [213]:
def dense_search(query: str, k: int = 3, max_distance: float | None = None) -> list[tuple[float, str]]:
    """Search the FAISS index and return top-k (distance, sentence) tuples.
    
    Lower distance = more similar (L2 distance, not similarity score).
    """
    # YOUR CODE HERE
    print("\nIn function dense_search")
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(query_embedding, k)

    # take first position as this is made for batch and not single
    distances = distances[0]
    indices = indices[0]
    print("distances ", distances)
    print("indices ", indices)

    result = []
    for i in range(k):
        index_of_sentence = indices[i]
        print("\nindex_of_sentence ",index_of_sentence)

        sentence = sentences[index_of_sentence]
        distance_of_sentence = distances[i]

        print("sentence = ",sentence)
        print("distance_of_sentence = ",distance_of_sentence)

        if max_distance is not None and distance_of_sentence > max_distance:
            continue

        result.append((distance_of_sentence, sentence))

    print("\nreturning this result : ",result)
    return result


### Exercise 1.5 — Test dense search

Run your search function with the query `"how precise was the science in this film"`.

**Expected result:** The top results should mention Kip Thorne, the Nobel Prize, scientific accuracy, and the black hole simulation — even though the query uses the word "precise" and the documents use "accurate", "scientifically", "Thorne".

This is the key insight: dense retrieval finds *meaning*, not *exact words*.

In [214]:
QUERY = "how precise was the science in this film"

results = dense_search(QUERY, k=3)

print(f"\nQuery: {QUERY!r}")
for rank, (dist, sentence) in enumerate(results, start=1):
    print(f"Rank {rank} | Distance: {dist:.4f}")
    print(f"  {sentence}")
    print()


In function dense_search
distances  [0.58424085 0.61065644 0.7151481 ]
indices  [12  8 11]

index_of_sentence  12
sentence =  The result was the first physically accurate simulation of a black hole ever created for a film.
distance_of_sentence =  0.58424085

index_of_sentence  8
sentence =  The film portrays the gravitational time dilation effect predicted by Einstein's general theory of relativity.
distance_of_sentence =  0.61065644

index_of_sentence  11
sentence =  The visual effects team worked with Thorne to produce scientifically accurate imagery of the black hole.
distance_of_sentence =  0.7151481

returning this result :  [(np.float32(0.58424085), 'The result was the first physically accurate simulation of a black hole ever created for a film.'), (np.float32(0.61065644), "The film portrays the gravitational time dilation effect predicted by Einstein's general theory of relativity."), (np.float32(0.7151481), 'The visual effects team worked with Thorne to produce scientifically 

### Exercise 1.6 — When Dense Retrieval Fails (The Always-Returns Problem)

Dense retrieval has a critical failure mode: it **always returns k results**, even when the query is completely outside the corpus. FAISS measures geometric distance in embedding space — it can only find the "least far away" vectors. It has no concept of "nothing relevant exists here."

Run a query about something entirely absent from the Interstellar corpus and observe what gets returned.

**What you will see:** Results look superficially plausible (sentences about space, large numbers) but are semantically wrong. Crucially, the **L2 distances will be much higher** than your in-corpus query from Ex 1.5 — that gap is the threshold signal.

**The production fix:** Add a `max_distance` threshold to `dense_search`. Any result with `distance > threshold` is filtered out. If nothing survives, return an empty list — the honest answer.

**Tasks:**
1. Run `dense_search("What is the mass of the moon?", k=3)` and print results with distances
2. Compare average distances with your in-corpus query — observe the gap
3. Go back to your `dense_search` (Ex 1.4) and add an optional `max_distance: float = None` parameter
4. Pick a threshold from the distance comparison, then verify the OOC query returns 0 results

In [215]:
# ---- 1. Out-of-corpus query ----
OOC_QUERY = "What is the mass of the moon?"

ooc_results = dense_search(OOC_QUERY, k=3)
ic_results  = dense_search(QUERY, k=3)       # QUERY defined in Ex 1.5

print(f"OUT-OF-CORPUS: {OOC_QUERY!r}")
print("-" * 70)
for rank, (dist, sent) in enumerate(ooc_results, 1):
    print(f"  Rank {rank} | Distance: {dist:.1f}  \u2192  {sent[:70]}")

print()
print(f"IN-CORPUS: {QUERY!r}")
print("-" * 70)
for rank, (dist, sent) in enumerate(ic_results, 1):
    print(f"  Rank {rank} | Distance: {dist:.1f}  \u2192  {sent[:70]}")

# ---- 2. Compare average distances ----
print()
avg_ooc = sum(d for d, _ in ooc_results) / len(ooc_results)
avg_ic  = sum(d for d, _ in ic_results)  / len(ic_results)
print(f"Avg distance \u2014 out-of-corpus : {avg_ooc:.1f}")
print(f"Avg distance \u2014 in-corpus     : {avg_ic:.1f}")
print("Notice: OOC distances are much larger. That gap is your threshold signal.")

# ---- 3 & 4. Add max_distance to dense_search ----
# TODO: Go back to your dense_search function (Ex 1.4) and add an optional parameter:
#
#   def dense_search(query: str, k: int = 3, max_distance: float = None):
#       ... (existing implementation) ...
#       if max_distance is not None:
#           results = [(d, s) for d, s in results if d <= max_distance]
#       return results
#
# Then uncomment and test:
# pick based on the distance gap above
threshold = 0.92
safe = dense_search(OOC_QUERY, k=3, max_distance=threshold)
print(f"With threshold={threshold}: {len(safe)} result(s) \u2014 expected 0")


In function dense_search
distances  [0.9137293 1.0264008 1.0409875]
indices  [10  9 30]

index_of_sentence  10
sentence =  Gargantua is a fictional supermassive rotating black hole with a mass 100 million times that of the Sun.
distance_of_sentence =  0.9137293

index_of_sentence  9
sentence =  On the water planet Miller, one hour equals seven years on Earth due to proximity to the massive black hole Gargantua.
distance_of_sentence =  1.0264008

index_of_sentence  30
sentence =  He sends the quantum data needed to solve the gravity equation back to his daughter Murph.
distance_of_sentence =  1.0409875

returning this result :  [(np.float32(0.9137293), 'Gargantua is a fictional supermassive rotating black hole with a mass 100 million times that of the Sun.'), (np.float32(1.0264008), 'On the water planet Miller, one hour equals seven years on Earth due to proximity to the massive black hole Gargantua.'), (np.float32(1.0409875), 'He sends the quantum data needed to solve the gravity equa

---

## Part 2 — Keyword Search: BM25

BM25 is the classic keyword search algorithm — a smarter version of word counting. It asks: does this document contain the query words? How rare are those words across the whole corpus? It has no idea that "precise" and "accurate" mean similar things — it only sees characters.

We will run the same query through BM25 and compare side by side with dense retrieval.

**Reference:** notes §2d — why BM25 fails the science query

### Exercise 2.1 — Build a BM25 index

Build a BM25 index from the `sentences` list.

**How BM25Okapi expects input:** It takes a list of *tokenized* documents — each document is a list of lowercase words, not a string.

**Hint:**
```python
tokenized = [sentence.lower().split() for sentence in sentences]
bm25 = BM25Okapi(tokenized)
```

In [216]:
# YOUR CODE HERE
tokenized_corpus = [sentence.lower().split() for sentence in sentences]
print("tokenized_corpus : ",tokenized_corpus)
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index built over {len(tokenized_corpus)} documents.")

tokenized_corpus :  [[], ['interstellar', 'is', 'a', '2014', 'epic', 'science', 'fiction', 'film', 'directed', 'by', 'christopher', 'nolan.'], ['the', 'screenplay', 'was', 'written', 'by', 'jonathan', 'nolan', 'and', 'christopher', 'nolan,', 'based', 'on', 'a', 'story', 'developed', 'by', 'jonathan', 'nolan.'], ['the', 'film', 'stars', 'matthew', 'mcconaughey,', 'anne', 'hathaway,', 'jessica', 'chastain,', 'and', 'michael', 'caine.'], ['it', 'was', 'produced', 'by', 'paramount', 'pictures', 'and', 'warner', 'bros.', 'pictures.'], ['the', 'story', 'follows', 'a', 'group', 'of', 'astronauts', 'who', 'travel', 'through', 'a', 'wormhole', 'near', 'saturn', 'in', 'search', 'of', 'a', 'new', 'home', 'for', 'humanity.'], ['theoretical', 'physicist', 'kip', 'thorne,', 'who', 'won', 'the', '2017', 'nobel', 'prize', 'in', 'physics,', 'served', 'as', 'executive', 'producer', 'and', 'scientific', 'consultant.'], ['thorne', 'ensured', 'that', 'the', 'depictions', 'of', 'relativity,', 'wormholes,', 

### Exercise 2.2 — Run BM25 and compare side-by-side

Run the same query `QUERY` through BM25 and print its top-3 results next to the dense retrieval results from Part 1.

**How to query BM25:**
1. Tokenize the query: `query_tokens = QUERY.lower().split()`
2. Get scores: `scores = bm25.get_scores(query_tokens)` — returns an array of length `num_sentences`
3. Sort by score descending and take top-k indices

**Question to think about:** Which results does BM25 return? Why does it fail to find the science-accuracy sentences? What word in the query leads it astray?

**Hint (notes §2d):** The query contains the word "science" — BM25 will latch onto that word and find documents where "science" appears literally, not documents about scientific accuracy.

In [217]:
def bm25_search(query: str, k: int = 3) -> list[tuple[float, str]]:
    """Search with BM25 and return top-k (score, sentence) tuples."""
    # YOUR CODE HERE
    print("\nIn function bm25_search")
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    # print("scores = ",scores)

    # using - negative sign since argsort is ascending sort by default. Using negative to get descending sort. ALso this will return indices
    sorted_scores = np.argsort(-scores)
    print("sorted_scores : ",sorted_scores)

    top_k_index = sorted_scores[:k]
    print("top_k_index : ",top_k_index)

    result = []

    # for i in range(k):
    #     index = top_k_index[i]
    #     score = scores[index]
    #     sentence = sentences[index]
    #     result.append((score, sentence))

    # or another way
    for index in top_k_index:
        score = scores[index]
        sentence = sentences[index]
        result.append((score, sentence))

    return result

bm25_results = bm25_search(QUERY, k=3)
dense_results = dense_search(QUERY, k=3)

print(f"\nQuery: {QUERY!r}")
print()

print("DENSE RETRIEVAL")
for rank, (dist, sent) in enumerate(dense_results, 1):
    print(f"  Rank {rank} | Distance: {dist:.4f}")
    print(f"  {sent}")
    print()

print("BM25 KEYWORD SEARCH")
for rank, (score, sent) in enumerate(bm25_results, 1):
    print(f"  Rank {rank} | Score: {score:.4f}")
    print(f"  {sent}")
    print()




In function bm25_search
sorted_scores :  [36  1 12 35  8  3  2 34  4 16  6  5 28 17 13 11 30 14  7  9 24 32 29 37
 22 23 31 10 33  0 19 26 25 21 20 18 15 27 38]
top_k_index :  [36  1 12]

In function dense_search
distances  [0.58424085 0.61065644 0.7151481 ]
indices  [12  8 11]

index_of_sentence  12
sentence =  The result was the first physically accurate simulation of a black hole ever created for a film.
distance_of_sentence =  0.58424085

index_of_sentence  8
sentence =  The film portrays the gravitational time dilation effect predicted by Einstein's general theory of relativity.
distance_of_sentence =  0.61065644

index_of_sentence  11
sentence =  The visual effects team worked with Thorne to produce scientifically accurate imagery of the black hole.
distance_of_sentence =  0.7151481

returning this result :  [(np.float32(0.58424085), 'The result was the first physically accurate simulation of a black hole ever created for a film.'), (np.float32(0.61065644), "The film portrays th

---

## Part 3 — Reranking with a Cross-Encoder

Dense retrieval is fast but imprecise — it embeds query and document *separately*, so it never directly compares them. A **cross-encoder** (also called a reranker) reads the query and document *together* as one input and outputs a single relevance score. Much more accurate, but slower — you can't pre-compute document scores.

The production pattern is a two-stage pipeline:
1. **Retrieve** a large candidate set cheaply (BM25 or dense, top-10 or top-100)
2. **Rerank** that small set expensively with the cross-encoder, return top-3

**Reference:** notes §3a–3b

### Exercise 3.1 — Load the cross-encoder

Load `CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')` from Sentence Transformers.

This model was fine-tuned on the MS MARCO passage ranking dataset to produce a relevance score for (query, passage) pairs. The score is a raw logit — higher is better.

In [218]:
# YOUR CODE HERE
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Cross-encoder reranker loaded.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7780.68it/s]


Cross-encoder reranker loaded.


### Exercise 3.2 — Write a `rerank` function

Write a function that takes a query and a list of candidate sentences, scores each (query, sentence) pair with the cross-encoder, and returns the candidates sorted by relevance score descending.

**How `CrossEncoder.predict` works:**
```python
pairs = [(query, sent) for sent in candidates]
scores = reranker.predict(pairs)   # returns array of float scores
```

Return a list of `(score, sentence)` tuples, sorted by score descending.

In [219]:
def rerank(query: str, candidates: list[str]) -> list[tuple[float, str]]:
    """Score each (query, candidate) pair with the cross-encoder.
    
    Returns list of (score, sentence) sorted by score descending.
    Higher score = more relevant.
    """
    # YOUR CODE HERE
    pairs = [(query, candidate) for candidate in candidates]
    print("pairs : ",pairs)
    scores = reranker.predict(pairs)
    print("scores : ",scores)

    result = sorted(zip(scores, candidates), reverse=True)
    return result

### Exercise 3.3 — BM25 → Reranker pipeline

Build the two-stage pipeline:
1. Retrieve top-10 candidates from BM25
2. Rerank those 10 with the cross-encoder
3. Return the top-3 from the reranker

Print the final top-3 results with their cross-encoder scores.

**Why use BM25 in stage 1 here?** Because we want to demonstrate that even with BM25's poor candidates, the reranker can still improve precision. In production you might use dense retrieval for stage 1 instead.

**Expected outcome:** The reranker should promote the science-relevant sentences to the top, even though BM25 retrieved mostly irrelevant candidates.

In [220]:
# Stage 1: BM25 retrieves 10 candidates
# YOUR CODE HERE
bm25_candidates = bm25_search(QUERY, k=10)
print("bm25_candidates : ",bm25_candidates)
candidates = [candidate for _, candidate in bm25_candidates]
print("candidates : ",candidates)

# Stage 2: Cross-encoder reranks those 10, return top-3
# YOUR CODE HERE
reranked = rerank(QUERY, candidates)

print(f"Query: {QUERY!r}")
print("\nBM25 → Reranker pipeline (top-3):")
print("-" * 60)
for rank, (score, sentence) in enumerate(reranked[:3], start=1):
    print(f"Rank {rank} | Score: {score:.4f}")
    print(f"  {sentence}")
    print()


In function bm25_search
sorted_scores :  [36  1 12 35  8  3  2 34  4 16  6  5 28 17 13 11 30 14  7  9 24 32 29 37
 22 23 31 10 33  0 19 26 25 21 20 18 15 27 38]
top_k_index :  [36  1 12 35  8  3  2 34  4 16]
bm25_candidates :  [(np.float64(5.577654479582324), 'It was released in 70mm film and IMAX formats as well as standard formats.'), (np.float64(5.251804313785585), 'Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.'), (np.float64(2.919603823217308), 'The result was the first physically accurate simulation of a black hole ever created for a film.'), (np.float64(2.7000416058098313), 'Interstellar premiered at the TCL Chinese Theatre in Hollywood on October 26, 2014.'), (np.float64(2.6670535593833415), "The film portrays the gravitational time dilation effect predicted by Einstein's general theory of relativity."), (np.float64(2.5807760806289304), 'The film stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, and Michael Caine.'), (np.float64(2.522

### Exercise 3.4 — Explicit Hybrid Pipeline: BM25 → Cross-Encoder

Combining the two stages you built is the **standard production retrieval pattern**:

- **Stage 1 (BM25):** Fast, cheap, casts a wide net over many candidates
- **Stage 2 (Cross-encoder):** Slow, precise, re-scores only those candidates with full query-document attention

BM25 is used as the first stage (rather than dense retrieval) because it runs in microseconds with zero GPU cost — making it cheap to over-retrieve 10–100 candidates. The cross-encoder does the precision work on that smaller set.

Write `hybrid_search` that wires these two stages together, then compare it side-by-side with raw BM25 on the same query.

In [221]:
def hybrid_search(query: str, top_k: int = 3, num_candidates: int = 10) -> list:
    """Two-stage retrieval: BM25 first stage followed by cross-encoder reranker.

    Args:
        query:          The search query string.
        top_k:          Final number of results to return after reranking.
        num_candidates: Number of BM25 candidates to retrieve in stage 1.
    Returns:
        List of (cross_encoder_score, sentence) tuples, sorted by score descending.
    """
    # Stage 1: BM25 retrieves a broad candidate set (cheap, fast)
    # YOUR CODE HERE
    bm25_hits = bm25_search(query, k=num_candidates)   # call bm25_search(query, k=num_candidates)
    print("bm25_hits : ",bm25_hits)

    # Extract just the sentence strings for the reranker
    # YOUR CODE HERE
    candidates = [sentence for _,sentence in bm25_hits]  # [sent for _, sent in bm25_hits]
    print("candidates : ",candidates)

    # Stage 2: Cross-encoder reranks only the candidates (slow, precise)
    # YOUR CODE HERE
    reranked = rerank(query, candidates)    # call rerank(query, candidates)
    print("reranked : ",reranked)
    
    return reranked[:top_k]


# ---- Compare raw BM25 vs hybrid ----
bm25_top3   = bm25_search(QUERY, k=3)
hybrid_top3 = hybrid_search(QUERY, top_k=3, num_candidates=10)

print(f"Query: {QUERY!r}\n")
print(f"{'BM25 (raw score)':<55}  |  HYBRID (BM25 \u2192 Cross-Encoder score)")
print("-" * 115)
for (bscore, bsent), (hscore, hsent) in zip(bm25_top3, hybrid_top3):
    b = bsent[:52] + "..." if len(bsent) > 52 else bsent
    h = hsent[:52] + "..." if len(hsent) > 52 else hsent
    print(f"[{bscore:.3f}] {b:<55}  |  [{hscore:.3f}] {h}")
print()
print("Does the hybrid pipeline surface more science-relevant sentences than raw BM25?")


In function bm25_search
sorted_scores :  [36  1 12 35  8  3  2 34  4 16  6  5 28 17 13 11 30 14  7  9 24 32 29 37
 22 23 31 10 33  0 19 26 25 21 20 18 15 27 38]
top_k_index :  [36  1 12]

In function bm25_search
sorted_scores :  [36  1 12 35  8  3  2 34  4 16  6  5 28 17 13 11 30 14  7  9 24 32 29 37
 22 23 31 10 33  0 19 26 25 21 20 18 15 27 38]
top_k_index :  [36  1 12 35  8  3  2 34  4 16]
bm25_hits :  [(np.float64(5.577654479582324), 'It was released in 70mm film and IMAX formats as well as standard formats.'), (np.float64(5.251804313785585), 'Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.'), (np.float64(2.919603823217308), 'The result was the first physically accurate simulation of a black hole ever created for a film.'), (np.float64(2.7000416058098313), 'Interstellar premiered at the TCL Chinese Theatre in Hollywood on October 26, 2014.'), (np.float64(2.6670535593833415), "The film portrays the gravitational time dilation effect predicted by Eins

---

## Part 4 — Retrieval Evaluation Metrics

How do you know if your retrieval system is actually good? You need a test suite: a corpus, a set of queries, and **relevance judgments** — a human-labelled answer to "which sentences are relevant for this query?".

We will implement three metrics that measure different aspects of quality:
- **Precision@k** — of the top-k results, what fraction are relevant?
- **Average Precision (AP)** — does ranking position matter? (yes, it does)
- **Mean Average Precision (MAP)** — AP averaged across many queries

**Reference:** notes §4a–4d  
All exercises in this part are **pure Python** — no models needed.

---

### Relevance judgments for the Interstellar corpus

Below is a hand-labelled relevance dictionary. Each key is a query string. Each value is a list of sentence indices (0-based) that are considered relevant for that query.

In [222]:
# Hand-labelled relevance judgments
# key   = query string
# value = list of sentence indices (into `sentences`) that are relevant

RELEVANCE = {
    "how precise was the science in this film": [5, 6, 7, 10, 11],
    "who directed interstellar":                [0],
    "what happens at the end of the film":      [30, 31, 32, 33, 34],
}

# Verify the indices make sense
print("Relevance check:")
for query, indices in RELEVANCE.items():
    print(f"\nQuery: {query!r}")
    for idx in indices:
        print(f"  [{idx}] {sentences[idx]}")

Relevance check:

Query: 'how precise was the science in this film'
  [5] The story follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humanity.
  [6] Theoretical physicist Kip Thorne, who won the 2017 Nobel Prize in Physics, served as executive producer and scientific consultant.
  [7] Thorne ensured that the depictions of relativity, wormholes, and black holes were as accurate as possible given the narrative constraints.
  [10] Gargantua is a fictional supermassive rotating black hole with a mass 100 million times that of the Sun.
  [11] The visual effects team worked with Thorne to produce scientifically accurate imagery of the black hole.

Query: 'who directed interstellar'
  [0] 

Query: 'what happens at the end of the film'
  [30] He sends the quantum data needed to solve the gravity equation back to his daughter Murph.
  [31] Murph uses the equation to enable humanity to leave Earth and populate space stations.
  [32] Cooper is ev

### Exercise 4.1 — Precision@k

**Formula:**
$$\text{Precision@k} = \frac{\text{number of relevant documents in top-k}}{k}$$

**Example (from notes §4b):**
```
Results (indices): [5, 0, 6]    Relevant indices: {5, 6, 7, 10, 11}

P@1 = 1/1 = 1.0   (index 5 is relevant)
P@2 = 1/2 = 0.5   (index 0 is not relevant)
P@3 = 2/3 = 0.67  (index 6 is relevant)
```

Write `precision_at_k`. It receives a list of retrieved sentence indices and a set of relevant indices.

In [223]:
def precision_at_k(retrieved_ids: list[int], relevant_ids: set[int], k: int) -> float:
    """Fraction of top-k retrieved items that are relevant.
    
    Args:
        retrieved_ids: Ordered list of retrieved sentence indices (rank 1 first).
        relevant_ids:  Set of indices that are ground-truth relevant.
        k:             Cutoff rank.
    Returns:
        Precision@k as a float in [0, 1].
    """
    # YOUR CODE HERE
    num_sum = 0
    for num in retrieved_ids[:k]:
        if num in relevant_ids:
            num_sum += 1
    
    print("num_sum : ",num_sum)
    prec_k = num_sum / k
    print("prec_k : ",prec_k)

    return prec_k

# Quick test
test_retrieved = [5, 0, 6, 10, 99]
test_relevant  = {5, 6, 7, 10, 11}

print("Precision@k test:")
for k in [1, 2, 3, 4, 5]:
    p = precision_at_k(test_retrieved, test_relevant, k)
    print(f"  P@{k} = {p:.4f}")

Precision@k test:
num_sum :  1
prec_k :  1.0
  P@1 = 1.0000
num_sum :  1
prec_k :  0.5
  P@2 = 0.5000
num_sum :  2
prec_k :  0.6666666666666666
  P@3 = 0.6667
num_sum :  3
prec_k :  0.75
  P@4 = 0.7500
num_sum :  3
prec_k :  0.6
  P@5 = 0.6000


### Exercise 4.2 — Average Precision (AP)

Precision@k does not care *where* in the list the relevant documents appear. AP rewards systems that rank relevant documents higher by computing precision only at positions where a relevant document appears, then averaging.

**Formula:**
$$\text{AP} = \frac{1}{R} \sum_{k=1}^{n} \text{Precision@k} \times \mathbb{1}[\text{doc}_k \text{ is relevant}]$$

where $R$ = total number of relevant documents that *exist in the archive* (not just in the top-k).

**Dry-run (from notes §4c):**
```
retrieved = [5, 0, 6]    relevant = {5, 6, 7, 10, 11}    R = 5

k=1: doc 5 is relevant  → P@1 = 1/1 = 1.00  → count it
k=2: doc 0 not relevant → skip
k=3: doc 6 is relevant  → P@3 = 2/3 = 0.67  → count it

AP = (1.00 + 0.67) / 5 = 0.334
```

**Important:** Divide by `R` (total relevant in archive), not by how many you retrieved.

In [224]:
def average_precision(retrieved_ids: list[int], relevant_ids: set[int]) -> float:
    """Average Precision for a single query.
    
    Args:
        retrieved_ids: Ordered list of retrieved sentence indices (rank 1 first).
        relevant_ids:  Set of all relevant indices that exist in the archive.
    Returns:
        Average Precision as a float in [0, 1].
    """
    # YOUR CODE HERE
    sum = 0
    precision_list = []
    for index, num  in enumerate(retrieved_ids, start=1):
        print("\nnum : ",num)
        if num in relevant_ids:
            sum = sum+1
            print(f"sum+1 : {sum} and index : {index}")

            prec_k = sum / (index)
            print("prec_k : ",prec_k)
            precision_list.append(prec_k)
            print("precision_list : ",precision_list)

    num_sum = 0
    for num in precision_list:
        num_sum+=num

    print("\nnum_sum : ",num_sum)
    print("R = ",len(relevant_ids))
    
    res = num_sum / len(relevant_ids)
    print("res : ",res)
    return res


# Quick test — matches the dry-run above
ap = average_precision([5, 0, 6], {5, 6, 7, 10, 11})
print(f"AP test (expected ~0.334): {ap:.4f}")

# Perfect ranking: relevant doc at position 1
ap_perfect = average_precision([5], {5})
print(f"\nAP perfect (expected 1.0): {ap_perfect:.4f}")

# Worst ranking: only relevant doc at position 3
ap_worst = average_precision([0, 99, 5], {5})
print(f"\nAP worst   (expected 0.33): {ap_worst:.4f}")


num :  5
sum+1 : 1 and index : 1
prec_k :  1.0
precision_list :  [1.0]

num :  0

num :  6
sum+1 : 2 and index : 3
prec_k :  0.6666666666666666
precision_list :  [1.0, 0.6666666666666666]

num_sum :  1.6666666666666665
R =  5
res :  0.3333333333333333
AP test (expected ~0.334): 0.3333

num :  5
sum+1 : 1 and index : 1
prec_k :  1.0
precision_list :  [1.0]

num_sum :  1.0
R =  1
res :  1.0

AP perfect (expected 1.0): 1.0000

num :  0

num :  99

num :  5
sum+1 : 1 and index : 3
prec_k :  0.3333333333333333
precision_list :  [0.3333333333333333]

num_sum :  0.3333333333333333
R =  1
res :  0.3333333333333333

AP worst   (expected 0.33): 0.3333


### Exercise 4.3 — Mean Average Precision (MAP)

MAP is just AP averaged across all queries in your test suite.

**Formula:**
$$\text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q)$$

**Hint:** You need a retrieval function to produce `retrieved_ids` for each query. Use your existing `dense_search` or `bm25_search` — but you need the *indices* not the sentences. Write a small helper or modify your search to also return indices.

In [225]:
def mean_average_precision(
    results_per_query:  dict[str, list[int]],
    relevant_per_query: dict[str, set[int]]
) -> float:
    """MAP across all queries.
    
    Args:
        results_per_query:  {query: [retrieved_sentence_indices]} in rank order.
        relevant_per_query: {query: {relevant_sentence_indices}}.
    Returns:
        MAP as a float in [0, 1].
    """
    # YOUR CODE HERE
    ap_sum = 0
    for key, values in results_per_query.items():
        print("\nkey : ", key)
        retrieved_ids = values
        print("retrieved_ids : ",retrieved_ids)
        relevant_ids = relevant_per_query[key]
        print("relevant_ids : ",relevant_ids)
        ap = average_precision(retrieved_ids, relevant_ids)
        ap_sum += ap 

    print("ap_sum : ",ap_sum)
    print("Q : ",len(results_per_query))

    result = ap_sum / len(results_per_query)
    return result

# Quick sanity check with made-up numbers
fake_results   = {"q1": [0, 1, 2], "q2": [0, 1, 2], "q3": [0, 1, 2]}
fake_relevant  = {"q1": {0},       "q2": {0, 1},    "q3": {2}}
# AP(q1) = 1.0, AP(q2) = (1.0 + 1.0)/2 = 1.0, AP(q3) = (1/3)/1 = 0.33
# MAP = (1.0 + 1.0 + 0.33) / 3 = 0.778
map_test = mean_average_precision(fake_results, fake_relevant)
print(f"MAP sanity check (expected ~0.778): {map_test:.4f}")


key :  q1
retrieved_ids :  [0, 1, 2]
relevant_ids :  {0}

num :  0
sum+1 : 1 and index : 1
prec_k :  1.0
precision_list :  [1.0]

num :  1

num :  2

num_sum :  1.0
R =  1
res :  1.0

key :  q2
retrieved_ids :  [0, 1, 2]
relevant_ids :  {0, 1}

num :  0
sum+1 : 1 and index : 1
prec_k :  1.0
precision_list :  [1.0]

num :  1
sum+1 : 2 and index : 2
prec_k :  1.0
precision_list :  [1.0, 1.0]

num :  2

num_sum :  2.0
R =  2
res :  1.0

key :  q3
retrieved_ids :  [0, 1, 2]
relevant_ids :  {2}

num :  0

num :  1

num :  2
sum+1 : 1 and index : 3
prec_k :  0.3333333333333333
precision_list :  [0.3333333333333333]

num_sum :  0.3333333333333333
R =  1
res :  0.3333333333333333
ap_sum :  2.3333333333333335
Q :  3
MAP sanity check (expected ~0.778): 0.7778


### Exercise 4.4 — Compare dense vs. BM25

Now use your MAP function on real retrieval results. For each query in `RELEVANCE`:
1. Get the top-5 retrieved indices from `dense_search`
2. Get the top-5 retrieved indices from `bm25_search`
3. Compute MAP for each system
4. Print a comparison table

**Hint:** Your search functions return `(distance/score, sentence)` tuples. You need to recover the original *index* of each sentence. The simplest way is to look it up: `sentences.index(sentence)`. (For ties, this picks the first occurrence — acceptable for this exercise.)

**Question to think about:** Which system gets a higher MAP? Why?

In [226]:
# Build retrieved_ids dicts for each system
# YOUR CODE HERE
dense_results_per_query = {}   # {query: [top-5 sentence indices]}
bm25_results_per_query  = {}   # {query: [top-5 sentence indices]}

relevant_per_query = {q: set(idxs) for q, idxs in RELEVANCE.items()}

print("")
for query in RELEVANCE:
    dense_hits = dense_search(query, k=5)
    print("dense_hits : ",dense_hits)

    bm25_hits = bm25_search(query, k=5)
    print("bm25_hits : ",bm25_hits)

    # long form code
    dense_top_k_sentence_indices = []
    for _, sentence in dense_hits:
        print("sentence of dense hits : ",sentence)
        index_of_sentence = sentences.index(sentence)
        print("index_of_sentence = ",index_of_sentence)
        dense_top_k_sentence_indices.append(index_of_sentence)

    print("dense_top_k_sentence_indices : ",dense_top_k_sentence_indices)

    # short form code
    bm25_top_k_sentence_indices = [sentences.index(sentence) for _,sentence in bm25_hits]
    print("bm25_top_k_sentence_indices : ",bm25_top_k_sentence_indices)

    dense_results_per_query[query] = dense_top_k_sentence_indices
    bm25_results_per_query[query] = bm25_top_k_sentence_indices


print("---- Final dense_results_per_query : ",dense_results_per_query)
print("---- Final bm25_results_per_query : ", bm25_results_per_query)

# Compute MAP
# YOUR CODE HERE
dense_map = mean_average_precision(dense_results_per_query, relevant_per_query)
bm25_map  = mean_average_precision(bm25_results_per_query, relevant_per_query)

print("dense_map : ",dense_map)
print("bm25_map : ", bm25_map)

print("\nRetrieval Evaluation (top-5, 3 queries)")
print("-" * 40)
print(f"Dense Retrieval MAP : {dense_map:.4f}")
print(f"BM25 Keyword   MAP  : {bm25_map:.4f}")
print()
print("Interpretation:")
if dense_map > bm25_map:
    print("Dense retrieval wins — meaning-based search handles these queries better.")
else:
    print("BM25 wins — exact keyword matching works well for these queries.")



In function dense_search
distances  [0.58424085 0.61065644 0.7151481  0.76904905 0.7809174 ]
indices  [12  8 11  1  7]

index_of_sentence  12
sentence =  The result was the first physically accurate simulation of a black hole ever created for a film.
distance_of_sentence =  0.58424085

index_of_sentence  8
sentence =  The film portrays the gravitational time dilation effect predicted by Einstein's general theory of relativity.
distance_of_sentence =  0.61065644

index_of_sentence  11
sentence =  The visual effects team worked with Thorne to produce scientifically accurate imagery of the black hole.
distance_of_sentence =  0.7151481

index_of_sentence  1
sentence =  Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.
distance_of_sentence =  0.76904905

index_of_sentence  7
sentence =  Thorne ensured that the depictions of relativity, wormholes, and black holes were as accurate as possible given the narrative constraints.
distance_of_sentence =  0.7809174

r

---

## Part 5 — Basic RAG with Ollama

RAG = Retrieval-Augmented Generation. Instead of asking the LLM to answer from memory (where it might hallucinate), you first retrieve the relevant passages from your corpus and then *stuff them into the prompt* before the question. The LLM reads from the page, not from fuzzy training memory.

You will call Ollama using the `requests` library — no LangChain, no wrappers. Raw HTTP.

**Ollama API endpoint:** `POST {OLLAMA_BASE_URL}/api/generate`  
**Request body:** `{"model": model_name, "prompt": prompt_string, "stream": false}`  
**Response field:** `response.json()["response"]`

**Reference:** notes §5a–5b

### Exercise 5.1 — Write `ollama_generate`

Write a function that sends a prompt to your local Ollama instance and returns the generated text as a string.

**Hint:**
```python
url = f"{OLLAMA_BASE_URL}/api/generate"
payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}
response = requests.post(url, json=payload)
```

Set `stream: false` so you get one complete JSON response instead of a streaming token-by-token response.

In [227]:
def ollama_generate(prompt: str) -> str:
    """Send a prompt to Ollama and return the generated text.
    
    Uses OLLAMA_BASE_URL and OLLAMA_MODEL from the config cell.
    """
    # YOUR CODE HERE
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        "model" : OLLAMA_MODEL,
        "prompt" : prompt,
        "stream" : False
    }

    response = requests.post(url, json = payload)

    return response.json()["response"]


# Smoke test — if Ollama is running, this should return a short answer
test_response = ollama_generate("In one sentence: what is a black hole?")
print(f"Ollama smoke test:")
print(test_response)

Ollama smoke test:
A black hole is a region in space where gravity is so strong that not even light can escape, formed from the remnants of massive stars after they undergo gravitational collapse, creating an area with such intense density and gravity that it warps spacetime beyond known thresholds.


### Exercise 5.2 — Build a RAG prompt

Write a function that assembles a grounded prompt from a question and a list of context chunks.

**Why context goes BEFORE the question (notes §5b — Lost in the Middle):**  
LLMs attend better to text at the start and end of the context window. Putting context before the question ensures the model processes the facts before it reads the question — reducing the chance it ignores them.

**Prompt template:**
```
Use ONLY the information below to answer the question. 
If the answer is not in the context, say "I don't know".

Context:
- {chunk_1}
- {chunk_2}
- {chunk_3}

Question: {question}

Answer:
```

In [228]:
def build_rag_prompt(question: str, context_chunks: list[str]) -> str:
    """Assemble a grounded RAG prompt with context before the question.
    
    Args:
        question:       The user's question.
        context_chunks: List of retrieved sentences to use as context.
    Returns:
        A complete prompt string ready to send to the LLM.
    """
    chunks = "\n".join(f"- {chunk}" for chunk in context_chunks)

    prompt = f"""Use ONLY the information below to answer the question.
If the answer is not in the context, say "I don't know".

Context:
{chunks}

Question: {question}

Answer:"""

    return prompt
    


# Preview what the assembled prompt looks like
sample_chunks = ["Kip Thorne served as scientific consultant.",
                 "The visual effects were scientifically accurate."]
sample_prompt = build_rag_prompt("How accurate was the science?", sample_chunks)
print("Assembled prompt:")
print("=" * 60)
print(sample_prompt)

Assembled prompt:
Use ONLY the information below to answer the question.
If the answer is not in the context, say "I don't know".

Context:
- Kip Thorne served as scientific consultant.
- The visual effects were scientifically accurate.

Question: How accurate was the science?

Answer:


### Exercise 5.3 — Write the full `rag` function

Combine everything into one pipeline function:
1. Retrieve the top-k sentences using `dense_search`
2. Extract just the sentence strings from the results
3. Build the grounded prompt with `build_rag_prompt`
4. Send it to Ollama with `ollama_generate`
5. Return the answer string

In [229]:
def rag(question: str, k: int = 3) -> str:
    """Full RAG pipeline: retrieve → build grounded prompt → generate.
    
    Args:
        question: The user's question.
        k:        Number of chunks to retrieve.
    Returns:
        The LLM's grounded answer.
    """
    # YOUR CODE HERE
    top_k_dense_res = dense_search(question, k)
    # print("top_k_dense_res : ",top_k_dense_res)
    top_k_sentences = [sentence for _, sentence in top_k_dense_res]
    print("top_k_sentences : ",top_k_sentences)
    
    rag_prompt = build_rag_prompt(question, top_k_sentences)
    print("rag_prompt ",rag_prompt)

    response_from_ollama = ollama_generate(rag_prompt)
    print("response_from_ollama : ",response_from_ollama)

    return response_from_ollama

In [230]:
# ---- Test: WITH OUT RAG ----
QUESTION = "How scientifically accurate is Interstellar and who ensured that?"

print("WITHOUT RAG (LLM answers from memory):")
print("-" * 60)
no_rag_answer = ollama_generate(QUESTION)
print(no_rag_answer)

WITHOUT RAG (LLM answers from memory):
------------------------------------------------------------
"Interstellar," a film directed by Christopher Nolan, incorporates various concepts from astrophysics and theoretical physics for its storyline involving wormholes and time travel. However, it's important to note that the portrayal of these scientific elements is more artistic license than strict adherence to actual science:

1. Wormholes as depicted in "Interstellar": The film introduces a concept similar to what theoretical physicists call 'Einstein-Rosen bridges' or Einstein-Rosen tunnels, which are hypothetical features of spacetime predicted by the general theory of relativity. Wormholes themselves have not been observed and remain speculative constructs that require exotic matter with negative energy to stabilize – something currently unknown in our understanding of physics.

2. Time dilation: The film's representation aligns more closely with real-world predictions from general re

In [231]:
# ---- Test: WITH RAG ----
print()
print("WITH RAG (LLM answers from retrieved context):")
print("-" * 60)
rag_answer = rag(QUESTION, k=3)
print(rag_answer)


WITH RAG (LLM answers from retrieved context):
------------------------------------------------------------

In function dense_search
distances  [0.5887574  0.6267158  0.67098725]
indices  [15  1 35]

index_of_sentence  15
sentence =  Interstellar received positive reviews from critics, who praised its ambition, visual effects, and performances.
distance_of_sentence =  0.5887574

index_of_sentence  1
sentence =  Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.
distance_of_sentence =  0.6267158

index_of_sentence  35
sentence =  Interstellar premiered at the TCL Chinese Theatre in Hollywood on October 26, 2014.
distance_of_sentence =  0.67098725

returning this result :  [(np.float32(0.5887574), 'Interstellar received positive reviews from critics, who praised its ambition, visual effects, and performances.'), (np.float32(0.6267158), 'Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.'), (np.float32(0.67098725), 'Interstellar 

In [232]:
# Test with RAG

test_questions = [
    "Who directed Interstellar?",
    # "Who stars in Interstellar?",
    # "What was special about the black hole in Interstellar?",
    "How much money did Interstellar make?",
    # "What Academy Award did Interstellar win?"
]

for question in test_questions:
    print(f"\nQ: {question}")
    print("-" * 60)
    answer = rag(question, k=10)
    print(f"A: {answer}")
    print()



Q: Who directed Interstellar?
------------------------------------------------------------

In function dense_search
distances  [0.34859282 0.5058185  0.5367433  0.6859834  0.84243655 0.85450137
 0.88345325 0.8862266  0.88969576 0.9104592 ]
indices  [ 1 35 15  2 18  6 19  3 12 11]

index_of_sentence  1
sentence =  Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.
distance_of_sentence =  0.34859282

index_of_sentence  35
sentence =  Interstellar premiered at the TCL Chinese Theatre in Hollywood on October 26, 2014.
distance_of_sentence =  0.5058185

index_of_sentence  15
sentence =  Interstellar received positive reviews from critics, who praised its ambition, visual effects, and performances.
distance_of_sentence =  0.5367433

index_of_sentence  2
sentence =  The screenplay was written by Jonathan Nolan and Christopher Nolan, based on a story developed by Jonathan Nolan.
distance_of_sentence =  0.6859834

index_of_sentence  18
sentence =  Cooper, played b

---

## Part 6 — LangChain RAG Pipeline

Everything you built in Part 5 by hand — embedding, FAISS indexing, prompt assembly, and the Ollama HTTP call — LangChain wraps into composable **chain** objects. This part re-builds the same RAG pipeline using LangChain so you can compare the raw approach against a production framework.

**Why learn LangChain if you already built the raw version?**
- Real codebases use frameworks — understanding the abstractions lets you debug them faster
- `RetrievalQA` and `PromptTemplate` appear throughout the LLM ecosystem
- Swapping components (different LLM, different vector store) becomes a one-line change

**Extra install required (if you haven't run it yet):**
```bash
pip install langchain langchain-community langchain-huggingface
```

**Reference:** Book Chapter 8 — "Example: RAG with Local Models" section

In [233]:
%pip install langchain langchain-community langchain-huggingface


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Exercise 6.1 — HuggingFace Embeddings + FAISS Vector Store via LangChain

LangChain wraps `sentence-transformers` via `HuggingFaceEmbeddings`. Instead of manually calling `model.encode()` and `index.add()`, you pass the embedding wrapper to `FAISS.from_texts()` and it handles everything internally.

**Note:** This builds a second, LangChain-managed FAISS index alongside the raw one from Part 1. Both index the same `sentences` list — you will see the results are equivalent.

After building the store, call `lc_db.similarity_search("how precise was the science", k=3)` and print the results. Each result is a `Document` object with `.page_content` (the text).

In [234]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS as LangChainFAISS

# TODO: Create a HuggingFaceEmbeddings wrapper for 'BAAI/bge-small-en-v1.5'
# YOUR CODE HERE
lc_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# TODO: Build the FAISS vector store from `sentences` in one call
# Hint: LangChainFAISS.from_texts(sentences, lc_embeddings)
# YOUR CODE HERE
lc_db = LangChainFAISS.from_texts(sentences, lc_embeddings)

# TODO: Verify it works — run a similarity search and print the results
results = lc_db.similarity_search("how precise was the science", k=3)
for doc in results:
    print(doc.page_content)

print("LangChain FAISS vector store ready.")
print(f"Vectors stored: {lc_db.index.ntotal}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8318.05it/s]


The result was the first physically accurate simulation of a black hole ever created for a film.
Thorne ensured that the depictions of relativity, wormholes, and black holes were as accurate as possible given the narrative constraints.
Theoretical physicist Kip Thorne, who won the 2017 Nobel Prize in Physics, served as executive producer and scientific consultant.
LangChain FAISS vector store ready.
Vectors stored: 39


### Exercise 6.2 — Ollama LLM Wrapper

LangChain's `Ollama` class wraps your local Ollama server as a standard LLM component. Once wrapped, you can pass it into any chain — LangChain handles the HTTP call, retries, and output parsing.

**Compare with Part 5:** In Part 5 you wrote `ollama_generate()` as a raw `requests.post()` call. Here LangChain handles that for you. The trade-off: convenience vs. full visibility into what happens under the hood. Knowing the raw version means you can always debug what LangChain is doing internally.

In [235]:
from langchain_community.llms import Ollama

# TODO: Create an Ollama LLM wrapper pointing at your local server
# Hint: Ollama(base_url=OLLAMA_BASE_URL, model=OLLAMA_MODEL)
# YOUR CODE HERE
lc_llm = Ollama(base_url=OLLAMA_BASE_URL, model=OLLAMA_MODEL)

# TODO: Smoke test — invoke the LLM and print the answer
answer = lc_llm.invoke("In one sentence: what is Retrieval-Augmented Generation?")
print("Smoke test:", answer)

print("LangChain Ollama LLM wrapper ready.")

Smoke test: Retrieval-augmented generation is a method in AI where systems dynamically pull relevant information from external data sources during text creation to enhance the accuracy, relevancy, and contextuality of generated content.
LangChain Ollama LLM wrapper ready.


### Exercise 6.3 — PromptTemplate

LangChain's `PromptTemplate` is a reusable, named-variable template. You define it once and any chain instantiates it by passing keyword arguments. This is equivalent to your `build_rag_prompt` from Part 5, but it is a first-class object that chains can inspect and validate automatically.

The template below uses the same "context before question" structure from Part 5 — context goes first to counteract the Lost-in-the-Middle problem.

In [236]:
from langchain.prompts import PromptTemplate

TEMPLATE = (
    "Use ONLY the information below to answer the question.\n"
    "If the answer is not in the context, say \"I don't know\".\n"
    "\n"
    "Context:\n"
    "{context}\n"
    "\n"
    "Question: {question}\n"
    "\n"
    "Answer:"
)

# TODO: Create the PromptTemplate object
# Hint: PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])
# YOUR CODE HERE
lc_prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

# Preview the template
sample = lc_prompt.format(
    context="Kip Thorne served as scientific consultant and won the 2017 Nobel Prize.",
    question="Who ensured the science was accurate?"
)
print("Template variables:", lc_prompt.input_variables)
print()
print("Sample instantiation:")
print(sample)

Template variables: ['context', 'question']

Sample instantiation:
Use ONLY the information below to answer the question.
If the answer is not in the context, say "I don't know".

Context:
Kip Thorne served as scientific consultant and won the 2017 Nobel Prize.

Question: Who ensured the science was accurate?

Answer:


### Exercise 6.4 — RetrievalQA Chain

`RetrievalQA` is LangChain's standard RAG chain. `chain_type='stuff'` means it "stuffs" all retrieved documents into the prompt at once — the simplest and most common approach for small corpora.

Wire together: `lc_db.as_retriever()` → `lc_prompt` → `lc_llm`, then invoke it and compare the answer to your raw Part 5 implementation on the same question. They should be similar — the difference is that LangChain is orchestrating the three steps for you.

In [237]:
from langchain.chains import RetrievalQA

# TODO: Build the RetrievalQA chain
# Hint:
#   rag_chain = RetrievalQA.from_chain_type(
#       llm=lc_llm,
#       chain_type="stuff",
#       retriever=lc_db.as_retriever(search_kwargs={"k": 3}),
#       chain_type_kwargs={"prompt": lc_prompt},
#       verbose=True,
#   )
# YOUR CODE HERE
rag_chain = RetrievalQA.from_chain_type(
    llm=lc_llm,
    chain_type="stuff",
    retriever=lc_db.as_retriever(search_kwargs={"k":3}),
    chain_type_kwargs={"prompt":lc_prompt},
    verbose=True
)

QUESTION = "How scientifically accurate is Interstellar and who ensured that?"

In [238]:
# TODO: Invoke the chain and print the answer
result = rag_chain.invoke({"query": QUESTION})
print("LangChain RAG answer:")
print(result["result"])

# Then compare to your Part 5 answer:
print("\nRaw Ollama RAG answer (Part 5):")
print(rag(QUESTION, k=3))



> Entering new RetrievalQA chain...

> Finished chain.
LangChain RAG answer:
The context provided does not contain specific information regarding the scientific accuracy of Interstellar or who ensured it. Therefore, I do not know based on the given context. To answer this question accurately, additional details about experts involved in verifying the film's adherence to real-world science would be required.

Raw Ollama RAG answer (Part 5):

In function dense_search
distances  [0.5887574  0.6267158  0.67098725]
indices  [15  1 35]

index_of_sentence  15
sentence =  Interstellar received positive reviews from critics, who praised its ambition, visual effects, and performances.
distance_of_sentence =  0.5887574

index_of_sentence  1
sentence =  Interstellar is a 2014 epic science fiction film directed by Christopher Nolan.
distance_of_sentence =  0.6267158

index_of_sentence  35
sentence =  Interstellar premiered at the TCL Chinese Theatre in Hollywood on October 26, 2014.
distance_of_s

### Deep Dive — `RetrievalQA.from_chain_type` Parameters Explained

When you call `RetrievalQA.from_chain_type(...)`, LangChain wires together your retriever, prompt, and LLM into a single callable pipeline. Here is what every parameter does.

---

#### `llm=lc_llm`

The language model that generates the final answer. LangChain wraps Ollama, OpenAI, HuggingFace, and others behind the same `BaseLLM` interface, so the chain does not care which backend you use.

---

#### `chain_type="stuff"`

Controls how retrieved documents are combined before being sent to the LLM. There are four options:

| `chain_type` | What it does | When to use |
|---|---|---|
| `"stuff"` | Stuffs all retrieved docs into one prompt | Small k, short docs — fits in context window |
| `"map_reduce"` | Sends each doc to LLM separately, then combines | Large k or long docs that won't fit together |
| `"refine"` | Starts with doc 1, iteratively refines with each new doc | When answer needs to accumulate across docs |
| `"map_rerank"` | Scores each doc independently, picks highest-scoring answer | When docs are competing, not complementary |

`"stuff"` is the default for short corpora. It is equivalent to your manual `build_rag_prompt` function — it joins all retrieved sentences and stuffs them into the `{context}` slot.

---

#### `retriever=lc_db.as_retriever(search_kwargs={"k": 3})`

`.as_retriever()` converts the FAISS vector store into a LangChain `BaseRetriever` — a standard interface the chain knows how to call. `search_kwargs={"k": 3}` is forwarded to the underlying `similarity_search` call. Change `k` here to retrieve more candidates (useful for the hubness problem — see notes on section 2e).

---

#### `chain_type_kwargs={"prompt": lc_prompt}`

Injects your custom `PromptTemplate` into the chain. Without this, LangChain uses its own default prompt which you cannot control. The dict is forwarded to the specific chain type — for `"stuff"`, the key that matters is `"prompt"`.

---

#### `verbose=True`

Prints intermediate steps — what the retriever fetched, what prompt was built, what the LLM received. Useful for debugging. Set to `False` in production.

---

#### How `input_variables` flow through the chain

Your `PromptTemplate` declares `input_variables=["context", "question"]`. These are the named slots in the template string `{context}` and `{question}`. When you call `rag_chain.invoke({"query": QUESTION})`, the chain fills them automatically:

```
invoke({"query": QUESTION})
    │
    ├─► retriever fetches top-k sentences
    │       → joined into one string → fills {context}
    │
    ├─► QUESTION string → fills {question}
    │
    └─► lc_prompt.format(context=..., question=...) → sent to LLM
```

The names `context` and `question` are the **contract** between your template and LangChain's stuff chain internals. If you rename `{context}` to `{docs}`, LangChain cannot find where to inject the retrieved sentences and raises a `KeyError`.

---

#### What if you need more than two variables?

| Scenario | Solution |
|---|---|
| Variable fixed forever (e.g. language, persona) | Use `partial_variables` in `PromptTemplate` — baked in at creation time |
| Variable filled by retriever | Name it `context` — filled automatically |
| Variable filled by your query | Name it `question` — filled automatically |
| Any other dynamic variable per call | Use LCEL (LangChain Expression Language) — see demo cell below |

`RetrievalQA` only knows how to fill `context` and `question`. For a third dynamic variable, you need LCEL which lets you build the chain step by step and pass any variables you want.

In [239]:

# DEMO: RAG with a custom extra variable using LCEL
# Shows how to go beyond the two built-in variables (context, question)
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Custom prompt with THREE variables ---
# context   → filled by retriever
# question  → filled by user input
# tone      → filled by user input (our custom extra variable)
custom_prompt = PromptTemplate(
    input_variables=["context", "question", "tone"],
    template="""Answer the question in a {tone} tone.
Use ONLY the information below. If the answer is not in the context, say 'I don't know'.

Context:
{context}

Question: {question}

Answer:"""
)

print("Custom prompt template:")
print(custom_prompt.template)
print()

# --- Helper to format retrieved docs into a single string ---
def format_docs(docs):
    return "\n".join(f"- {doc.page_content}" for doc in docs)

# --- Build LCEL chain ---
# Each key in the dict maps to one input_variable in the prompt
lcel_chain = (
    {
        "context":  lc_db.as_retriever(search_kwargs={"k": 5}) | format_docs,
        "question": RunnablePassthrough(),   # passes the user's question string through
        "tone":     RunnablePassthrough(),   # we'll inject this manually below
    }
    | custom_prompt
    | lc_llm
    | StrOutputParser()
)

# --- Invoke with all three variables ---
# NOTE: Because 'question' and 'tone' both use RunnablePassthrough(),
# we need a slightly different approach — pass a dict directly
from langchain_core.runnables import RunnableLambda

custom_chain = (
    {
        "context":  (lambda x: x["question"]) | lc_db.as_retriever(search_kwargs={"k": 5}) | format_docs,
        "question": RunnableLambda(lambda x: x["question"]),
        "tone":     RunnableLambda(lambda x: x["tone"]),
    }
    | custom_prompt
    | lc_llm
    | StrOutputParser()
)

print("Running LCEL chain with tone='casual'...")
print("-" * 60)
answer_casual = custom_chain.invoke({
    "question": "Who directed Interstellar?",
    "tone": "casual"
})
print(f"Casual: {answer_casual}")
print()

print("Running LCEL chain with tone='formal'...")
print("-" * 60)
answer_formal = custom_chain.invoke({
    "question": "Who directed Interstellar?",
    "tone": "formal"
})
print(f"Formal: {answer_formal}")


Custom prompt template:
Answer the question in a {tone} tone.
Use ONLY the information below. If the answer is not in the context, say 'I don't know'.

Context:
{context}

Question: {question}

Answer:

Running LCEL chain with tone='casual'...
------------------------------------------------------------
Casual: Christopher Nolan directed Interstellar. He's known for his ambitious projects that often involve complex storytelling and stunning visual effects!

Running LCEL chain with tone='formal'...
------------------------------------------------------------
Formal: Interstellar was directed by Christopher Nolan. This information can be derived from the provided context which states that "Interstellar is a 2 extraterrestrial science fiction film directed by Christopher Nolan." Therefore, it confirms that he took on the directorial role for this project in 2014.


### How the LCEL Custom Chain Works

The `custom_chain` above uses **LangChain Expression Language (LCEL)**. The `|` operator chains steps like a Unix pipe — output of each step becomes input of the next.

The chain receives one dict as input:

```python
{"question": "Who directed Interstellar?", "tone": "casual"}
```

The first step is a dict with three keys. Each key maps to one `{variable}` slot in the prompt template. All three run **in parallel** on the same input dict:

```python
{
    "context":  (lambda x: x["question"]) | lc_db.as_retriever(...) | format_docs,
    "question": RunnableLambda(lambda x: x["question"]),
    "tone":     RunnableLambda(lambda x: x["tone"]),
}
```

**`"context"` key** — three sub-steps chained together:
1. `lambda x: x["question"]` — extracts just the question string from the input dict
2. `lc_db.as_retriever(...)` — runs similarity search, returns `[Document, Document, ...]`
3. `format_docs` — joins them into `"- sentence1\n- sentence2\n..."`

**`"question"` key** — `RunnableLambda(lambda x: x["question"])` extracts the question string and passes it through unchanged.

**`"tone"` key** — `RunnableLambda(lambda x: x["tone"])` extracts the tone string and passes it through unchanged.

After the parallel step, the dict looks like:

```python
{
    "context":  "- Interstellar is a 2014 film...\n- ...",
    "question": "Who directed Interstellar?",
    "tone":     "casual"
}
```

Then the remaining pipe steps run sequentially:

```
| custom_prompt    → fills {context}, {question}, {tone} slots → full prompt string
| lc_llm           → sends prompt to Ollama → raw LLM output
| StrOutputParser() → strips wrapper objects → clean Python string
```

The key reason to use LCEL instead of `RetrievalQA` here is the third variable `{tone}`. `RetrievalQA` only knows how to fill `context` and `question` automatically. Any additional dynamic variable requires LCEL so you control exactly how each slot is populated.

---

## Part 7 — Advanced RAG

The basic RAG pipeline (retrieve → stuff → generate) works well when the user's question is clean and specific. Real questions are often messy, verbose, or compound. These two exercises tackle that.

**Reference:** notes §5e

### Exercise 7.1 — Query Rewriting

**The problem:** Users write conversational or verbose questions that confuse the retriever. For example:

> "Hey, I was just curious, I watched this movie last night, it was pretty cool, but like, I want to know — how accurate were the space things in it really?"

BM25 and even dense retrieval will struggle with that noise. The fix: ask the LLM to rewrite the question into a clean, specific search query *before* retrieval.

**Write `rewrite_query`:** Send the messy question to Ollama with a meta-prompt that asks it to produce only a clean search query (no explanation, no preamble). Then use the rewritten query in `dense_search`.

**Meta-prompt template:**
```
Rewrite the following question as a short, specific search query for a document retrieval system.
Return only the search query — no explanation, no preamble.

Question: {messy_question}

Search query:
```

In [242]:
def rewrite_query(messy_question: str) -> str:
    """Use the LLM to rewrite a verbose question into a clean search query.
    
    Returns only the cleaned query string.
    """
    # YOUR CODE HERE
    prompt = f"""Rewrite the following question as a short, specific search query for a document retrieval system.
Return only the search query — no explanation, no preamble.

Question: {messy_question}

Search query:
"""
    
    return lc_llm.invoke(prompt).strip()


def rag_with_rewriting(question: str, k: int = 3) -> str:
    """RAG pipeline with query rewriting before retrieval."""
    # YOUR CODE HERE
    rewritten = rewrite_query(question)
    print(f"Original : {question}")
    print(f"Rewritten : {rewrite_query}")

    hits = dense_search(rewritten, k=k)
    context = [sentence for _, sentence in hits]

    prompt = build_rag_prompt(question, context)
    return ollama_generate(prompt)

MESSY_QUESTION = (
    "Hey I just watched this space movie last night and I remember there was this "
    "physics guy involved, and I think the black hole looked really real? "
    "Like how scientifically correct was all of that exactly?"
)

clean_query = rewrite_query(MESSY_QUESTION)
print(f"Original: {MESSY_QUESTION}")
print(f"\nRewritten: {clean_query!r}")

print("\nRAG answer with rewritten query:")
print("-" * 60)
print(rag_with_rewriting(MESSY_QUESTION, k=3))

Original: Hey I just watched this space movie last night and I remember there was this physics guy involved, and I think the black hole looked really real? Like how scientifically correct was all of that exactly?

Rewritten: 'How accurate is special effects portrayal of a black hole in recent popular science fiction movies?'

RAG answer with rewritten query:
------------------------------------------------------------
Original : Hey I just watched this space movie last night and I remember there was this physics guy involved, and I think the black hole looked really real? Like how scientifically correct was all of that exactly?
Rewritten : <function rewrite_query at 0x11edc6ca0>

In function dense_search
distances  [0.45373613 0.54882133 0.5712361 ]
indices  [12  8 11]

index_of_sentence  12
sentence =  The result was the first physically accurate simulation of a black hole ever created for a film.
distance_of_sentence =  0.45373613

index_of_sentence  8
sentence =  The film portrays t

### Exercise 7.2 — Multi-Query RAG

**The problem:** Some questions contain two sub-questions. For example:

> "Compare the scientific accuracy and the box office performance of Interstellar."

A single retrieval step will return chunks that are biased toward one sub-topic. The fix: decompose the question into sub-queries, run each independently, deduplicate the retrieved chunks, then generate one unified answer.

**Steps:**
1. Use the LLM to decompose the question into a list of 2–3 sub-queries (one per line)
2. Run `dense_search` for each sub-query
3. Collect all retrieved sentences, deduplicate (preserve insertion order)
4. Build a grounded prompt with the combined context
5. Generate the answer

**Decomposition meta-prompt:**
```
Decompose the following question into 2-3 specific sub-queries for document retrieval.
Return one sub-query per line, no numbering, no explanation.

Question: {question}

Sub-queries:
```

In [248]:
def decompose_question(question: str) -> list[str]:
    """Use the LLM to decompose a compound question into sub-queries.
    
    Returns a list of query strings (one per sub-question).
    """
    # YOUR CODE HERE
    prompt = f"""Decompose the following question into 2-3 specific sub-queries for document retrieval.
Return one sub-query per line, no numbering, no explanation.

Question: {question}

Sub-queries:"""
    
    response = lc_llm.invoke(prompt).strip()
    print("decompose question raw: ",response)

    sub_queries = [line.strip() for line in response.splitlines() if line.strip()]
    print("decompose_question parsed : ",sub_queries)

    return sub_queries

def multi_query_rag(question: str, k_per_query: int = 3) -> str:
    """Multi-query RAG: decompose → retrieve per sub-query → deduplicate → generate.
    
    Args:
        question:     The compound question.
        k_per_query:  How many chunks to retrieve per sub-query.
    Returns:
        A unified answer grounded in all retrieved context.
    """
    # YOUR CODE HERE
    sub_queries = decompose_question(question)
    print("sub_queries : ",sub_queries)

    seen = set()
    all_contexts = []
    for sub_q in sub_queries:
        hits = dense_search(sub_q, k=k_per_query)
        for _, sentence in hits:
            if sentence not in seen:
                seen.add(sentence)
                all_contexts.append(sentence)

    print(f"\nUnique sentence retrieved: {len(all_contexts)}")

    prompt = build_rag_prompt(question, all_contexts)
    print(f"\nFinal prompt with context: ", prompt)
    return lc_llm.invoke(prompt)

COMPOUND_QUESTION = (
    "Compare the scientific accuracy and the box office performance of Interstellar."
)

sub_queries = decompose_question(COMPOUND_QUESTION)
print("Decomposed into sub-queries:")
for i, q in enumerate(sub_queries, 1):
    print(f"  {i}. {q}")

print("\nMulti-query RAG answer:")
print("-" * 60)
print(multi_query_rag(COMPOUND_QUESTION))

decompose question raw:  1. Retrieve scholarly articles or studies evaluating the scientific plausibility and theories presented in "Interstellar."
2. Search for data on the film's gross earnings at the global box office from credible financial databases.
decompose_question parsed :  ['1. Retrieve scholarly articles or studies evaluating the scientific plausibility and theories presented in "Interstellar."', "2. Search for data on the film's gross earnings at the global box office from credible financial databases."]
Decomposed into sub-queries:
  1. 1. Retrieve scholarly articles or studies evaluating the scientific plausibility and theories presented in "Interstellar."
  2. 2. Search for data on the film's gross earnings at the global box office from credible financial databases.

Multi-query RAG answer:
------------------------------------------------------------
decompose question raw:  1. Retrieve articles analyzing the critical reception and factual content in relation to space s

---

## Part 8 — Mini Projects

Choose **one** of the three projects below. Each one extends what you built in Parts 1–6 to a new dataset or evaluation task. These are open-ended — there is no single correct solution.

---

### Project A — Study Notes Q&A

**Goal:** Turn your own chapter 8 notes into a queryable knowledge base.

Load the chapter 8 notes markdown file, split it into paragraphs, embed them, build a FAISS index, and use your `rag` pipeline to answer questions about what you studied.

**Steps:**
1. Read the notes file into a string
2. Split by double newline (`\n\n`) to get paragraphs; filter out short ones (< 50 characters)
3. Embed the paragraphs and build a new FAISS index (separate from the Interstellar index)
4. Write a `notes_rag(question)` function that searches this notes index
5. Ask at least 3 questions about Chapter 8 content

**Example questions:**
- "What is the difference between a bi-encoder and a cross-encoder?"
- "Why does chunking with overlap help with boundary blindness?"
- "What is the lost in the middle problem?"

In [261]:
import json

# ollama generate token by token
def ollama_generate_token_wise(prompt: str) -> str:
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": True}

    response = requests.post(url, json=payload, stream=True)
    full_text = ""

    for line in response.iter_lines():
        if line:
            chunk = json.loads(line)
            token = chunk.get("response","")
            print(token, end="", flush=True)
            full_text += token
            if chunk.get("done"):
                break

    print()
    return full_text

In [256]:
# helper function to split paragraphs
def split_into_paragraphs(text: str):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    # filter out noise
    paragraphs = [
        p for p in paragraphs
        if not p.startswith("---")  # horizontal rules
        and not p.startswith("# ")  # top-level headings alone
        and not p.startswith("1. [")    # table of contents
        and len(p) > 50 # too short to be useful
    ]
    
    print(f"Total paragraphs : {len(paragraphs)}")

    return paragraphs

In [262]:
# ============================================================
# PROJECT A — Study Notes Q&A
# ============================================================

NOTES_PATH = "../../../notes/ch08-semantic-search-and-rag.md"  # adjust if needed

# YOUR CODE HERE
# 1. Load and chunk the notes
with open(NOTES_PATH, "r", encoding="utf-8") as f:
    notes_text = f.read()

paragraphs = split_into_paragraphs(notes_text)
print("paragraphs : ",paragraphs)

# 2. Embed paragraphs
notes_embeddings = embed_model.encode(paragraphs, show_progress_bar=True)
print(f"Embeddings Shape : ", notes_embeddings.shape)

# 3. Build FAISS index
notes_dim = notes_embeddings.shape[1]
notes_index = faiss.IndexFlatL2(notes_dim)
notes_index.add(np.float32(notes_embeddings))
print(f"FAISS index built with {notes_index.ntotal} vectors")

# 4. Search function for notes
def notes_search(query: str, k: int = 3) -> list[tuple[float, str]]:
    query_vec = embed_model.encode([query])
    distances, indices = notes_index.search(np.float32(query_vec), k)
    result = [(distances[0][i], paragraphs[indices[0][i]]) for i in range(k)]
    return result

# 5. RAG function for notes
def notes_rag(question: str, k: int=3) -> str:
    hits = notes_search(question, k=k)
    context = [para for _, para in hits]
    prompt = build_rag_prompt(question, context)
    print("\nFinal prompt : ",prompt)
    # return ollama_generate(prompt)
    return ollama_generate_token_wise(prompt)

# 6. Test it
print("---------- Final Test -------------")
print(notes_rag("What is the hubness problem in dense retrieval?"))
print(notes_rag("What is the difference between bi-encoder and cross-encoder?"))

Total paragraphs : 388
paragraphs :  ['> Search was the first killer application of language models, and RAG is the technique that tames their greatest weakness — hallucination — by grounding every answer in retrieved facts.', '## 1. Overview: The Three Pillars of LLM-Powered Search', 'Months after the publication of the original BERT paper in 2018, Google announced it was using transformer models to power Google Search, calling it "one of the biggest leaps forward in the history of Search." Microsoft Bing followed with similar announcements. The reason these mature, billion-user systems benefited so dramatically from language models is a capability called **semantic search** — the ability to retrieve documents by meaning rather than by exact word matching. A keyword search engine can find the word "science" in a document. A semantic search system understands that the phrase "how precise was the science" is asking about scientific accuracy, even if no document uses those exact words.',

Batches: 100%|██████████| 13/13 [00:01<00:00,  9.05it/s]


Embeddings Shape :  (388, 384)
FAISS index built with 388 vectors
---------- Final Test -------------

Final prompt :  Use ONLY the information below to answer the question.
If the answer is not in the context, say "I don't know".

Context:
- **The hubness problem.** In high-dimensional embedding spaces, certain sentences become *hubs* — points that are geometrically close to a disproportionately large number of query vectors. These hub sentences crowd out more specific, relevant sentences by winning the nearest-neighbour race for almost every query, even when they are not the best answer.
- **The problem:** Dense retrieval finds *locally* relevant chunks — chunks that are semantically close to the query. But it struggles with *global* questions that require synthesising information across the entire corpus. "What are the major themes across all these documents?" cannot be answered by any individual chunk, no matter how good the retrieval is.
- Dense retrieval is powerful but not perfe

---

### Project B — Recipe RAG

**Goal:** Build a recipe finder that retrieves recipes by ingredient or description.

Use the five recipes provided as string constants below. Embed them at the sentence level, build a FAISS index, and create a `recipe_rag(question)` function that answers ingredient and preparation questions.

**Example queries:**
- "What can I make with chickpeas and lemon?"
- "How do I make a quick pasta dish?"
- "What is a good recipe for someone who doesn't eat meat?"

In [267]:
RECIPES = [
    """Lemon Chickpea Soup.
    Ingredients: 2 cans chickpeas, 1 lemon (juice and zest), 4 cups vegetable broth, 1 onion, 3 garlic cloves, cumin, coriander, olive oil, salt and pepper.
    Instructions: Sauté onion and garlic in olive oil until soft. Add cumin and coriander. Add chickpeas and broth. Simmer 15 minutes. Stir in lemon juice and zest. Season with salt and pepper. Serve with crusty bread.
    This is a vegetarian soup, ready in about 25 minutes.""",

    """Spaghetti Aglio e Olio.
    Ingredients: 400g spaghetti, 6 garlic cloves (thinly sliced), half cup olive oil, red chili flakes, fresh parsley, parmesan cheese, salt.
    Instructions: Cook spaghetti until al dente. While pasta cooks, slowly fry garlic in olive oil until golden. Add chili flakes. Toss pasta with garlic oil and pasta water. Finish with parsley and parmesan.
    This is a classic Italian pasta dish ready in under 20 minutes. It is vegetarian.""",

    """Chicken Tikka Masala.
    Ingredients: 700g chicken breast, 1 cup yogurt, tikka masala spice mix, 2 cans crushed tomatoes, 1 cup heavy cream, onion, garlic, ginger, butter.
    Instructions: Marinate chicken in yogurt and spices for 1 hour. Grill or bake chicken. Sauté onion, garlic, ginger in butter. Add tomatoes and simmer. Add cream. Add grilled chicken. Serve with rice or naan.
    This is a popular Indian curry dish with chicken. It takes about 1.5 hours including marination.""",

    """Black Bean Tacos.
    Ingredients: 2 cans black beans, corn tortillas, cumin, smoked paprika, lime juice, avocado, salsa, sour cream, cheddar cheese, cilantro.
    Instructions: Drain and rinse beans. Cook beans with cumin and smoked paprika for 5 minutes. Warm tortillas. Assemble tacos with beans, sliced avocado, salsa, sour cream, cheese, and cilantro. Squeeze lime on top.
    This is a quick vegetarian taco recipe ready in 15 minutes.""",

    """Salmon with Honey Garlic Glaze.
    Ingredients: 4 salmon fillets, 3 tablespoons honey, 3 garlic cloves (minced), 2 tablespoons soy sauce, 1 tablespoon butter, lemon, fresh dill.
    Instructions: Mix honey, garlic, and soy sauce. Sear salmon in butter 3 minutes per side. Pour glaze over salmon. Cook 2 more minutes until glaze caramelises. Serve with lemon and dill.
    This is a quick seafood recipe ready in 15 minutes. It pairs well with steamed vegetables or rice.""",
]

# ============================================================
# PROJECT B — Recipe RAG
# ============================================================

# YOUR CODE HERE
# 1. Split each recipe into sentences and track which recipe each sentence belongs to
# no splitting required

# 2. Embed all sentences
recipe_embedding = embed_model.encode(RECIPES, show_progress_bar=True)
print(f"Embedded {len(RECIPES)} recipes")

# 3. Build FAISS index
recipe_dim = recipe_embedding.shape[1]
recipe_index = faiss.IndexFlatL2(recipe_dim)
recipe_index.add(np.float32(recipe_embedding))

# recipe search function
def recipe_search(query: str, k: int = 2) -> list[tuple[float, str]]:
    query_vec = embed_model.encode([query])
    distance, indices = recipe_index.search(np.float32(query_vec), k)
    print("distance : ",distance)
    print("indices : ",indices)
    res = [(distance[0][i], RECIPES[indices[0][i]]) for i in range(k)]

    return res

# 4. Write recipe_rag(question) function that answers questions about recipes
def recipe_rag(query: str):
    hits = recipe_search(query)
    print("hits : ",hits)
    context = [recipe for _,recipe in hits]
    print("context : ",context)

    prompt = build_rag_prompt(query, context)
    print("Final prompt : ",prompt)

    return ollama_generate_token_wise(prompt)

# 5. Test with at least 3 different ingredient/type queries
test_queries = [
    "What can I cook with chickpeas?",
    "I want a quick vegetarian dinner",
    "What seafood recipes do you have?",
]

for query in test_queries:
    print(f"Q: {query}")
    print(recipe_rag(query))
    print()



Batches: 100%|██████████| 1/1 [00:00<00:00,  4.02it/s]


Embedded 5 recipes
Q: What can I cook with chickpeas?
distance :  [[0.46689147 0.68201965]]
indices :  [[0 3]]
hits :  [(np.float32(0.46689147), 'Lemon Chickpea Soup.\n    Ingredients: 2 cans chickpeas, 1 lemon (juice and zest), 4 cups vegetable broth, 1 onion, 3 garlic cloves, cumin, coriander, olive oil, salt and pepper.\n    Instructions: Sauté onion and garlic in olive oil until soft. Add cumin and coriander. Add chickpeas and broth. Simmer 15 minutes. Stir in lemon juice and zest. Season with salt and pepper. Serve with crusty bread.\n    This is a vegetarian soup, ready in about 25 minutes.'), (np.float32(0.68201965), 'Black Bean Tacos.\n    Ingredients: 2 cans black beans, corn tortillas, cumin, smoked paprika, lime juice, avocado, salsa, sour cream, cheddar cheese, cilantro.\n    Instructions: Drain and rinse beans. Cook beans with cumin and smoked paprika for 5 minutes. Warm tortillas. Assemble tacos with beans, sliced avocado, salsa, sour cream, cheese, and cilantro. Squeeze 

---

### Project C — Evaluate Your RAG

**Goal:** Build a ground-truth test set for the Interstellar corpus and measure how well your `rag` function performs.

You will evaluate along two axes:
1. **Retrieval quality:** MAP of the dense_search results against your relevance judgments
2. **Answer faithfulness (manual):** For each question, read the RAG answer and judge whether it is grounded in the retrieved context or if it hallucinated

**Steps:**
1. Write 5 questions about the Interstellar corpus
2. For each question, manually identify which sentence indices are relevant (like `RELEVANCE` in Part 4)
3. Run `dense_search(question, k=5)` for each question and record the retrieved indices
4. Compute MAP across your 5 questions
5. Run `rag(question)` for each question and record the answers
6. For each answer: label it Faithful / Hallucinated / Partially faithful based on whether the answer is supported by the retrieved context
7. Print a summary table: question | MAP contribution | faithfulness label

In [268]:
# PROJECT C — Evaluate Your RAG

# Step 1: Write your 5 questions and relevance judgments
# MY_RELEVANCE = {
#     "your question here": [list_of_relevant_sentence_indices],
#     e.g.
#     "Who composed the Interstellar score?": [12, 13],
# }

MY_RELEVANCE = {
    "Who directed Interstellar?": [1],
    "Who ensured the science was accurate?": [6, 7, 11, 12],
    "How much did Interstellar gross?": [16],
    "What awards did Interstellar win?": [17],
    "What happens at the end of the film?": [30, 31, 32, 33, 34],
}


# YOUR CODE HERE
# 2. Run dense_search for each question
dense_results = {}
for questions in MY_RELEVANCE:
    hits = dense_search(question, k=5)
    dense_results[question] = [sentences.index(s) for _, s in hits]

# 3. Compute MAP
relevant_per_query = {q: set(idxs) for q, idxs in MY_RELEVANCE.items()}
map_score = mean_average_precision(dense_results, relevant_per_query)
print(f"MAP: {map_score:.4f}")

# 4. Run rag() for each question
# 5. Label each answer: Faithful / Hallucinated / Partially faithful
print("\nRAG Faithfulness Evaluation")
print("-" * 60)
for question in MY_RELEVANCE:
    answer = rag(question, k=5)
    print(f"Q: {question}")
    print(f"A: {answer}")
    label = input("Label (F=Faithful / H=Hallucinated / P=Partial): ")
    print()

# 6. Print summary table


In function dense_search
distances  [0.51181483 0.52125597 0.56123745 0.85300744 0.87549555]
indices  [35 15  1 16  2]

index_of_sentence  35
sentence =     - [5e. Advanced RAG Techniques](#5e-advanced-rag-techniques)
distance_of_sentence =  0.51181483

index_of_sentence  15
sentence =     - [2d. Dense vs. Keyword Search (BM25)](#2d-dense-vs-keyword-search-bm25)
distance_of_sentence =  0.52125597

index_of_sentence  1
sentence =  
distance_of_sentence =  0.56123745

index_of_sentence  16
sentence =     - [2e. Caveats of Dense Retrieval](#2e-caveats-of-dense-retrieval)
distance_of_sentence =  0.85300744

index_of_sentence  2
sentence =  ## Hands-On Large Language Models — Chapter 8
distance_of_sentence =  0.87549555

returning this result :  [(np.float32(0.51181483), '   - [5e. Advanced RAG Techniques](#5e-advanced-rag-techniques)'), (np.float32(0.52125597), '   - [2d. Dense vs. Keyword Search (BM25)](#2d-dense-vs-keyword-search-bm25)'), (np.float32(0.56123745), ''), (np.float32(0.8530

KeyError: 'How much money did Interstellar make?'

---

## Checkpoint — What You Built

By completing this notebook you have implemented, from scratch:

| Component | Tool Used | Part |
|-----------|-----------|------|
| Sentence splitting | Python `str.split` | 1.1 |
| Dense embeddings | `sentence-transformers` BAAI/bge-small | 1.2 |
| Exact nearest-neighbour search | `faiss.IndexFlatL2` | 1.3 |
| Dense search function | FAISS `index.search` | 1.4 |
| Dense retrieval failure mode + distance threshold | Manual distance comparison | 1.6 |
| Keyword search | `rank_bm25.BM25Okapi` | 2.1–2.2 |
| Cross-encoder reranker | `CrossEncoder` ms-marco-MiniLM | 3.1–3.2 |
| BM25 → reranker pipeline | Two-stage | 3.3 |
| Explicit hybrid search function | BM25 + CrossEncoder combined | 3.4 |
| Precision@k | Pure Python | 4.1 |
| Average Precision | Pure Python | 4.2 |
| Mean Average Precision | Pure Python | 4.3–4.4 |
| Ollama raw API call | `requests.post` | 5.1 |
| Grounded RAG prompt | String assembly | 5.2 |
| Full RAG pipeline (raw) | retrieve → prompt → generate | 5.3 |
| LangChain embeddings + FAISS | `HuggingFaceEmbeddings` + `FAISS.from_texts` | 6.1 |
| LangChain LLM wrapper | `langchain_community.llms.Ollama` | 6.2 |
| LangChain PromptTemplate | `PromptTemplate` | 6.3 |
| Full RAG pipeline (LangChain) | `RetrievalQA` chain | 6.4 |
| Query rewriting | LLM meta-prompt | 7.1 |
| Multi-query decomposition | LLM decomposition + deduplication | 7.2 |